# Reporting Automation Workflow

This notebook demonstrates the analytical review layer of the recurring reporting pipeline. It reconciles validation outputs, explains rejected records, summarizes approved transactions, and reviews invoice collection health.

**Workflow:** raw inputs → transformation → validation → KPI calculation → report and dashboard outputs.

All data is synthetic and intended for demonstration.

## 1. Setup and portable project discovery

In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)


def find_project_root(start: Path | None = None) -> Path:
    """Find the nearest parent containing the processed data directory."""
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data" / "processed").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate data/processed. Run this notebook inside the repository."
    )


PROJECT_ROOT = find_project_root()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
print(f"Project root: {PROJECT_ROOT}")

## 2. Load and validate reporting outputs

The notebook fails early when an expected output or required field is missing.

In [ ]:
FILES = {
    "transactions": "validated_transactions.csv",
    "validation_report": "validation_report.csv",
    "rejected": "rejected_records.csv",
    "invoice_status": "invoice_payment_status.csv",
}
missing_files = [name for name in FILES.values() if not (PROCESSED_DIR / name).is_file()]
if missing_files:
    raise FileNotFoundError(f"Missing processed outputs: {missing_files}")

transactions = pd.read_csv(PROCESSED_DIR / FILES["transactions"])
validation_report = pd.read_csv(PROCESSED_DIR / FILES["validation_report"])
rejected = pd.read_csv(PROCESSED_DIR / FILES["rejected"])
invoice_status = pd.read_csv(PROCESSED_DIR / FILES["invoice_status"])

required_transaction_columns = {
    "transaction_id", "customer_id", "business_unit", "region",
    "transaction_date", "amount", "validation_status", "validation_reason",
}
required_invoice_columns = {
    "invoice_id", "customer_id", "invoice_amount", "collected_amount",
    "outstanding_amount", "reporting_month", "is_paid", "is_overdue",
}
missing_transaction_columns = required_transaction_columns.difference(transactions.columns)
missing_invoice_columns = required_invoice_columns.difference(invoice_status.columns)
if missing_transaction_columns or missing_invoice_columns:
    raise ValueError(
        f"Missing required columns — transactions: {sorted(missing_transaction_columns)}, "
        f"invoices: {sorted(missing_invoice_columns)}"
    )

transactions["transaction_date"] = pd.to_datetime(
    transactions["transaction_date"], errors="raise"
)
transactions["amount"] = pd.to_numeric(transactions["amount"], errors="raise")
invoice_status["reporting_month"] = pd.to_datetime(
    invoice_status["reporting_month"], format="%Y-%m", errors="raise"
)
for column in ["invoice_amount", "collected_amount", "outstanding_amount"]:
    invoice_status[column] = pd.to_numeric(invoice_status[column], errors="raise")

transactions.head()

## 3. Validation reconciliation

The summary file, detailed transaction output, and rejected-record extract should agree exactly.

In [ ]:
detail_counts = (
    transactions["validation_status"].value_counts()
    .rename_axis("status")
    .rename("detail_count")
)
reported_counts = (
    validation_report.set_index("status")["count"]
    .rename("reported_count")
)
reconciliation = pd.concat([reported_counts, detail_counts], axis=1).fillna(0).astype(int)
reconciliation["difference"] = (
    reconciliation["detail_count"] - reconciliation["reported_count"]
)

rejected_detail_count = int(transactions["validation_status"].eq("REJECTED").sum())
duplicate_id_rows = transactions[transactions.duplicated("transaction_id", keep=False)]
duplicates_are_quarantined = (
    duplicate_id_rows.empty
    or (
        duplicate_id_rows["validation_status"].eq("REJECTED").all()
        and duplicate_id_rows["validation_reason"].eq("duplicate_transaction").all()
    )
)
checks = {
    "Validation report matches transaction detail": reconciliation["difference"].eq(0).all(),
    "Rejected extract matches transaction detail": len(rejected) == rejected_detail_count,
    "Duplicate transaction IDs are quarantined": duplicates_are_quarantined,
    "Rejected rows contain a reason": rejected["validation_reason"].notna().all(),
}
if not all(checks.values()):
    failed = [name for name, passed in checks.items() if not passed]
    raise ValueError(f"Reporting reconciliation failed: {failed}")

reconciliation

In [ ]:
validation_kpis = pd.Series({
    "Total records": len(transactions),
    "Approved records": int(transactions["validation_status"].eq("OK").sum()),
    "Rejected records": rejected_detail_count,
    "Approval rate (%)": 100 * transactions["validation_status"].eq("OK").mean(),
    "Rejection rate (%)": 100 * transactions["validation_status"].eq("REJECTED").mean(),
}, name="Value")
validation_kpis.to_frame()

## 4. Rejection analysis

In [ ]:
rejection_summary = (
    rejected.groupby("validation_reason", dropna=False)
    .agg(records=("transaction_id", "count"), affected_amount=("amount", "sum"))
    .sort_values("records", ascending=False)
    .reset_index()
)
rejection_summary["share_of_rejections_pct"] = (
    100 * rejection_summary["records"] / len(rejected)
)
rejection_summary.round(2)

## 5. Approved transaction reporting

In [ ]:
approved = transactions.loc[transactions["validation_status"].eq("OK")].copy()
approved["reporting_month"] = approved["transaction_date"].dt.to_period("M").dt.to_timestamp()

monthly_transactions = (
    approved.groupby("reporting_month", as_index=False)
    .agg(
        transactions=("transaction_id", "count"),
        customers=("customer_id", "nunique"),
        total_amount=("amount", "sum"),
        average_amount=("amount", "mean"),
    )
)
monthly_transactions["amount_change_pct"] = monthly_transactions["total_amount"].pct_change() * 100
monthly_transactions.round(2)

In [ ]:
business_unit_summary = (
    approved.groupby("business_unit")
    .agg(
        transactions=("transaction_id", "count"),
        customers=("customer_id", "nunique"),
        total_amount=("amount", "sum"),
        average_amount=("amount", "mean"),
    )
    .sort_values("total_amount", ascending=False)
)
business_unit_summary["amount_share_pct"] = (
    100 * business_unit_summary["total_amount"] / business_unit_summary["total_amount"].sum()
)
business_unit_summary.round(2)

## 6. Invoice collection health

In [ ]:
invoice_monthly = (
    invoice_status.groupby("reporting_month", as_index=False)
    .agg(
        invoices=("invoice_id", "count"),
        customers=("customer_id", "nunique"),
        invoiced_amount=("invoice_amount", "sum"),
        collected_amount=("collected_amount", "sum"),
        outstanding_amount=("outstanding_amount", "sum"),
        paid_invoices=("is_paid", "sum"),
        overdue_invoices=("is_overdue", "sum"),
    )
)
invoice_monthly["collection_rate_pct"] = (
    100 * invoice_monthly["collected_amount"] / invoice_monthly["invoiced_amount"]
)
invoice_monthly["overdue_invoice_rate_pct"] = (
    100 * invoice_monthly["overdue_invoices"] / invoice_monthly["invoices"]
)
invoice_monthly.round(2)

## 7. Reporting review checklist

- Confirm that all reconciliation checks pass before distributing reports.
- Investigate the largest rejection reason and whether it indicates a source-system issue.
- Review unusual month-over-month transaction movements.
- Compare business-unit contribution and customer coverage.
- Escalate declining collection rates or increasing overdue invoice rates.

The notebook reads existing pipeline outputs and does not modify source or processed data.